<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 📋 TODO 与 recitation：治「忘」

**一句话定位**：长任务里早期计划会被埋在 messages 顶部，注意力衰减（context rot）；让 LLM **周期性重写 TODO 列表**，等于把目标「复述到 context 末尾」。
> 对应主课 `1_a_🔥_todo.ipynb`。2026-10 现状：deepagents 0.7 起 `write_todos` **opt-in**。

</div>

### 1️⃣ 病：context rot；药：recitation

| | |
|---|---|
| 病因 | Manus 数据：平均一个任务 ~50 次工具调用；context 一长，LLM 对**最早**那条「用户想要什么」的注意力衰减 |
| 症状 | 跑题、遗忘目标、重复劳动、把中间结果当终点 |
| 药 | **recitation（复述）**：每完成一项就重写整个 TODO，新列表出现在 messages **末尾**，注意力被强制拉回 |
| 真实产品 | Claude Code 的 `TodoWrite`、Cursor Agent、ChatGPT Agent 都在生产里跑「写 TODO → 干活 → 复述 TODO」 |

一条 TODO 只有两个字段：`content`（具体可执行）+ `status`（`pending / in_progress / completed`）。

### 2️⃣ `write_todos` 的实现：本课 = 库，一字不差

In [ ]:
from typing import Annotated, Literal

from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.types import Command
from typing_extensions import TypedDict


class Todo(TypedDict):
    content: str
    status: Literal["pending", "in_progress", "completed"]


@tool
def write_todos(todos: list[Todo], tool_call_id: Annotated[str, InjectedToolCallId]) -> Command:
    """Create or update the agent's TODO list (full rewrite every time)."""
    return Command(update={
        "todos": todos,                                                     # 整体覆盖，不配 reducer
        "messages": [ToolMessage(f"Updated todo list to {todos}", tool_call_id=tool_call_id)],  # 回显 = 复述
    })


# 库里的版本（langchain.agents.middleware.todo）长得一模一样：
from langchain.agents.middleware.todo import write_todos as lib_write_todos

print(lib_write_todos.description[:220], "...")

### 3️⃣ 设计要点（为什么这样写）

| 要点 | 原因 |
|---|---|
| `todos` **不配 reducer**，整体覆盖 | Manus 风格要求 LLM **每次重写整个列表**，重写过程就是「重新考虑」 |
| ToolMessage 把列表**回显** | 列表同时出现在两处：LLM 自己发的 tool_call + 返回的 ToolMessage → 处于 context 末尾 |
| `read_todos`（本课有、库没有） | 它不是给程序员读的，是**给 LLM 自己读的自我提示工具**：把 `state["todos"]` 塞回 messages 末尾 |
| 工具描述写「何时用 / 何时不用」 | 库的描述明说：**< 3 步的琐事别用**；`write_todos` 不要并行调用 |
| `in_progress` 数量 | 本课：只能 1 个；库：允许多个**互不相关**的并行 |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

### 🆕 2026-10：`write_todos` 变成 opt-in 的来龙去脉

| 时间 | 状态 |
|---|---|
| ≤ 0.6（本课录制时 0.2.7） | `create_deep_agent` 默认带 `write_todos` + planning prompt |
| **0.7（2026-07-29）** | 官方 evals：planning prompt + `write_todos` **无显著增益** → 移出默认 harness；默认一轮基础 input token ~6k → ~2k（−65%）。`BASE_AGENT_PROMPT` 标记弃用（0.9 删除） |
| 0.7.28（现在） | `from langchain.agents.middleware import TodoListMiddleware`；`create_deep_agent(..., middleware=[TodoListMiddleware()])`；子 agent 要用得各自加 |

本机实测：不加 middleware 时模型工具列表 = `ls read_file write_file edit_file delete glob grep task`；加上后多出 `write_todos`，调用后 `state["todos"]` 即为传入列表，ToolMessage 为 `Updated todo list to [...]`。

</div>

### 4️⃣ planning ≠ delegation

| | Planning（`write_todos`） | Delegation（`task` → 子 agent） |
|---|---|---|
| 解决 | 主 agent 自己**忘** | 主 agent context 被**挤 / 乱** |
| 谁干活 | 主 agent 自己 | 另一个 agent，在隔离 context 里 |
| 产物 | state 里一张可见的白板 | 一条 ToolMessage 结果 |

新课的建议：TODO 用于 **≥ 3 步、可能需要修订计划**的任务；最后一条 `write_todos` 之后**再单独发一条**正文回答——「标记完成」本身不是答案。

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：TODO 治**忘**：`write_todos` = **整体重写 + ToolMessage 回显**，把目标复述到 context 末尾；0.7 起它是 **`TodoListMiddleware()` opt-in**，库里没有 `read_todos`，琐事别开。

</div>